In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error


# 1. LOAD + MERGE DATA
df_main = pd.read_csv("dataset.csv")
df_age = pd.read_csv("age_data.csv")

df = df_main.merge(df_age, on="Year", how="inner")
df = df.sort_values("Year").reset_index(drop=True)


# 2. AGE COLUMNS
age_cols = [col for col in df.columns if "Population 31 Dec" in col]
age_cols = sorted(age_cols, key=lambda x: int(x.split()[0]))


# 3. COMPRESS AGE STRUCTURE
df["Youth"] = df[age_cols[0:20]].sum(axis=1)
df["Working_Age"] = df[age_cols[20:65]].sum(axis=1)
df["Elderly"] = df[age_cols[65:]].sum(axis=1)

df["Dependency_ratio"] = df["Elderly"] / df["Working_Age"]


# 4. FEATURE ENGINEERING
df["Population_lag1"] = df["Population"].shift(1)
df["Population_lag2"] = df["Population"].shift(2)

df["Migration_lag1"] = df["Net migration"].shift(1)
df["Natural_lag1"] = df["Natural increase"].shift(1)

df["Population_delta"] = df["Population"].diff()

df = df.dropna().reset_index(drop=True)


# 5. FEATURES / TARGET
features = [
    'Year',
    'Net migration',
    'Natural increase',
    'Population_lag1',
    'Population_lag2',
    'Migration_lag1',
    'Natural_lag1',
    'Youth',
    'Working_Age',
    'Elderly',
    'Dependency_ratio'
]


# 6. TRAIN / TEST SPLIT
df_train = df[df["Year"] <= 2018]
df_test = df[(df["Year"] > 2018) & (df["Year"] <= 2024)]

X_train = df_train[features].values
X_test = df_test[features].values

y_train = df_train["Population_delta"].values
y_test = df_test["Population_delta"].values


# 7. SCALING
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))


# 8. ENSEMBLE OF BIG MODELS (for supercomputer) - FIXED VERSION
def create_model(architecture_id, features_len):
    
    if architecture_id == 2:  # Residual with bottlenecks - Use Functional API
        # Input layer
        inputs = tf.keras.layers.Input(shape=(features_len,))
        
        # Initial dense layer
        x = tf.keras.layers.Dense(1024, activation="relu")(inputs)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.3)(x)
        
        # Residual blocks
        for _ in range(6):
            residual = x
            x = tf.keras.layers.Dense(512, activation="relu")(x)
            x = tf.keras.layers.BatchNormalization()(x)
            x = tf.keras.layers.Dropout(0.3)(x)
            x = tf.keras.layers.Dense(512, activation="relu")(x)
            # Ensure residual has same shape as x
            if residual.shape[-1] != x.shape[-1]:
                residual = tf.keras.layers.Dense(512)(residual)
            x = tf.keras.layers.Add()([x, residual])
            x = tf.keras.layers.Activation("relu")(x)
        
        # Final layers
        x = tf.keras.layers.Dense(256, activation="relu")(x)
        x = tf.keras.layers.Dropout(0.3)(x)
        x = tf.keras.layers.Dense(128, activation="relu")(x)
        outputs = tf.keras.layers.Dense(1)(x)
        
        # Create model
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        return model
    
    else:  # architecture_id == 0, 1, or 3 - Use Sequential API
        model = tf.keras.Sequential()
        model.add(tf.keras.layers.Input(shape=(features_len,)))
        
        if architecture_id == 0:  # Wide & Deep
            model.add(tf.keras.layers.Dense(2048, activation="relu"))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.4))
            model.add(tf.keras.layers.Dense(1024, activation="relu"))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.4))
            model.add(tf.keras.layers.Dense(512, activation="relu"))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.3))
            model.add(tf.keras.layers.Dense(256, activation="relu"))
            model.add(tf.keras.layers.Dropout(0.3))
            model.add(tf.keras.layers.Dense(128, activation="relu"))
            
        elif architecture_id == 1:  # Very Deep
            for _ in range(12):
                model.add(tf.keras.layers.Dense(512, activation="relu"))
                model.add(tf.keras.layers.BatchNormalization())
                model.add(tf.keras.layers.Dropout(0.3))
            model.add(tf.keras.layers.Dense(256, activation="relu"))
            model.add(tf.keras.layers.Dropout(0.3))
            model.add(tf.keras.layers.Dense(128, activation="relu"))
            
        else:  # architecture_id == 3 - Ultra Wide
            model.add(tf.keras.layers.Dense(4096, activation="relu"))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.5))
            model.add(tf.keras.layers.Dense(2048, activation="relu"))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.5))
            model.add(tf.keras.layers.Dense(1024, activation="relu"))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.4))
            model.add(tf.keras.layers.Dense(512, activation="relu"))
        
        model.add(tf.keras.layers.Dense(1))
        return model

print("Training ensemble of 15 big models...")
ensemble_models = []
ensemble_predictions = []

# Train multiple models (can be parallelized on supercomputer)
for arch_id in range(15):  # 15 different architectures
    print(f"\nTraining model {arch_id+1}/15...")
    model = create_model(arch_id % 4, len(features))
    
    # Custom learning rate for each model (variety)
    lr = 0.001 * (0.5 ** (arch_id // 5))  # Decaying LR for some models
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )
    
    # Callbacks for each model
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=100,
        restore_best_weights=True
    )
    
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=30,
        min_lr=1e-7,
        verbose=0
    )
    
    # Train with different batch sizes
    batch_size = 64 * (2 ** (arch_id % 3))  # 64, 128, or 256
    
    history = model.fit(
        X_train_scaled,
        y_train_scaled,
        batch_size=batch_size,
        epochs=1000,
        validation_data=(X_test_scaled, y_test_scaled),
        callbacks=[early_stop, reduce_lr],
        verbose=0  # Set to 0 for cleaner output
    )
    
    ensemble_models.append(model)
    
    # Get predictions for ensemble
    pred_scaled = model.predict(X_test_scaled, verbose=0)
    ensemble_predictions.append(pred_scaled)
    
    # Print progress
    val_mae = min(history.history['val_mae'])
    print(f"  Model {arch_id+1} done. Best val MAE: {val_mae:.4f}")

# Ensemble prediction (average)
y_pred_scaled_ensemble = np.mean(ensemble_predictions, axis=0)

# Also train a mega-model (single best architecture)
print("\nTraining mega-model (best architecture from ensemble)...")
mega_model = create_model(0, len(features))  # Using architecture 0 as base

mega_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)

early_stop_mega = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=200,
    restore_best_weights=True
)

reduce_lr_mega = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=40,
    min_lr=1e-8,
    verbose=1
)

# Train mega-model longer
history_mega = mega_model.fit(
    X_train_scaled,
    y_train_scaled,
    batch_size=128,
    epochs=2000,
    validation_data=(X_test_scaled, y_test_scaled),
    callbacks=[early_stop_mega, reduce_lr_mega],
    verbose=1
)

# Plot mega-model training
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history_mega.history['loss'], label='train_loss')
plt.plot(history_mega.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Mega-Model Loss')
plt.yscale('log')

plt.subplot(1,2,2)
plt.plot(history_mega.history['mae'], label='train_mae')
plt.plot(history_mega.history['val_mae'], label='val_mae')
plt.legend()
plt.title('Mega-Model MAE')
plt.tight_layout()
plt.show()


# 10. BACKTEST (using ensemble AND mega-model)
# Ensemble backtest
y_pred_delta_ensemble = scaler_y.inverse_transform(y_pred_scaled_ensemble).flatten()

reconstructed_ensemble = []
current_pop = df.loc[df["Year"] == 2018, "Population"].values[0]

for delta in y_pred_delta_ensemble:
    current_pop += delta
    reconstructed_ensemble.append(current_pop)

mae_ensemble = mean_absolute_error(df_test["Population"], reconstructed_ensemble)

# Mega-model backtest
y_pred_scaled_mega = mega_model.predict(X_test_scaled)
y_pred_delta_mega = scaler_y.inverse_transform(y_pred_scaled_mega).flatten()

reconstructed_mega = []
current_pop = df.loc[df["Year"] == 2018, "Population"].values[0]

for delta in y_pred_delta_mega:
    current_pop += delta
    reconstructed_mega.append(current_pop)

mae_mega = mean_absolute_error(df_test["Population"], reconstructed_mega)

print(f"\nMAE (Ensemble): {mae_ensemble:,.0f}")
print(f"MAE (Mega-Model): {mae_mega:,.0f}")

# Use the better model for forecasting
if mae_ensemble < mae_mega:
    print("\nUsing Ensemble model for forecasting (better MAE)")
    best_model = ensemble_models
    use_ensemble = True
else:
    print("\nUsing Mega-Model for forecasting (better MAE)")
    best_model = mega_model
    use_ensemble = False


# 11. AGE MODEL SETUP
ages = np.arange(len(age_cols))
age_state = df[df["Year"] == 2024][age_cols].values[0].astype(float)

mortality_rate = np.zeros(len(ages))
for age in ages:
    if age < 1:
        mortality_rate[age] = 0.003
    elif age < 15:
        mortality_rate[age] = 0.0005
    elif age < 40:
        mortality_rate[age] = 0.001
    elif age < 65:
        mortality_rate[age] = 0.005
    elif age < 80:
        mortality_rate[age] = 0.02
    else:
        mortality_rate[age] = 0.08

fertility_rate = np.zeros(len(ages))
for age in range(15, 50):
    fertility_rate[age] = 0.04

avg_migration = df["Net migration"].tail(5).mean()
avg_natural = df["Natural increase"].tail(5).mean()

migration = np.zeros(len(ages))
migration[20:45] = avg_migration / 25

def simulate_year(state):
    new_state = np.zeros_like(state)

    for age in range(len(state) - 1, 0, -1):
        new_state[age] = state[age - 1] * (1 - mortality_rate[age - 1])

    births = np.sum(state * fertility_rate)
    new_state[0] = births

    new_state += migration

    return new_state


# 12. FORECAST (DYNAMIC AGE) - Using best model
future_years = np.arange(2025, 2041)

last = df[df["Year"] == 2024].iloc[0]
prev = df[df["Year"] == 2023].iloc[0]

current_population = last["Population"]

pop_lag1 = last["Population"]
pop_lag2 = prev["Population"]

migration_lag1 = last["Net migration"]
natural_lag1 = last["Natural increase"]

current_state = age_state.copy()

future_pred = []
future_pred_upper = []
future_pred_lower = []

for year in future_years:
    current_state = simulate_year(current_state)

    youth = np.sum(current_state[0:20])
    working = np.sum(current_state[20:65])
    elderly = np.sum(current_state[65:])
    dep_ratio = elderly / working if working > 0 else 0

    X_input = np.array([[  
        year,
        avg_migration,
        avg_natural,
        pop_lag1,
        pop_lag2,
        migration_lag1,
        natural_lag1,
        youth,
        working,
        elderly,
        dep_ratio
    ]])

    X_scaled = scaler_X.transform(X_input)
    
    if use_ensemble:
        # Ensemble prediction with uncertainty
        ensemble_deltas = []
        for model in ensemble_models:
            delta_scaled = model.predict(X_scaled, verbose=0)
            delta = scaler_y.inverse_transform(delta_scaled)[0][0]
            ensemble_deltas.append(delta)
        
        delta = np.mean(ensemble_deltas)
        delta_std = np.std(ensemble_deltas)
        delta_upper = delta + 1.96 * delta_std
        delta_lower = delta - 1.96 * delta_std
    else:
        # Mega-model prediction
        delta_scaled = best_model.predict(X_scaled, verbose=0)
        delta = scaler_y.inverse_transform(delta_scaled)[0][0]
        delta_upper = delta
        delta_lower = delta

    current_population += delta
    future_pred.append(current_population)
    
    if use_ensemble:
        future_pred_upper.append(current_population + 1.96 * delta_std)
        future_pred_lower.append(current_population - 1.96 * delta_std)
    else:
        future_pred_upper.append(current_population)
        future_pred_lower.append(current_population)

    pop_lag2 = pop_lag1
    pop_lag1 = current_population
    migration_lag1 = avg_migration
    natural_lag1 = avg_natural

future_pred = np.array(future_pred)
future_pred_upper = np.array(future_pred_upper)
future_pred_lower = np.array(future_pred_lower)


# 13. PLOTS
plt.figure(figsize=(14,7))

# Historical and backtest
plt.plot(df["Year"], df["Population"], 'b-', label="Historical", linewidth=2)
plt.plot(df_test["Year"], reconstructed_ensemble if use_ensemble else reconstructed_mega, 
         'g--o', label=f"Backtest ({'Ensemble' if use_ensemble else 'Mega-Model'})", markersize=4)

# Forecast with uncertainty
plt.plot(future_years, future_pred, 'r--o', label="Forecast", linewidth=2, markersize=6)

if use_ensemble:
    plt.fill_between(future_years, future_pred_lower, future_pred_upper, 
                     alpha=0.3, color='red', label='95% Confidence Interval')

plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.title("Population Forecast (Dynamic Age + ML Ensemble)", fontsize=14)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Population", fontsize=12)
plt.tight_layout()
plt.show()


# 14. OUTPUT
print("\n" + "="*60)
print(f"FORECAST 2025–2040 (using {'ENSEMBLE' if use_ensemble else 'MEGA-MODEL'})")
print("="*60)
print(f"{'Year':<10} {'Population':<15} {'Lower 95%':<15} {'Upper 95%':<15}")
print("-"*60)
for i, (y, p) in enumerate(zip(future_years, future_pred)):
    if use_ensemble:
        print(f"{y:<10} {p:>12,.0f}   {future_pred_lower[i]:>12,.0f}   {future_pred_upper[i]:>12,.0f}")
    else:
        print(f"{y:<10} {p:>12,.0f}   {'N/A':>12}   {'N/A':>12}")
print("="*60)

# Save results
results_df = pd.DataFrame({
    'Year': future_years,
    'Forecast': future_pred,
    'Lower_95': future_pred_lower if use_ensemble else future_pred,
    'Upper_95': future_pred_upper if use_ensemble else future_pred
})
results_df.to_csv('population_forecast_ensemble.csv', index=False)
print("\nResults saved to 'population_forecast_ensemble.csv'")